# Partes 3 e 5 — Vocabulários grandes com PCA e retrieval por imagem de consulta

Este notebook usa **os mesmos encodings e as mesmas partições** dos notebooks anteriores:

- `train` dos ficheiros → treino (80%) + validação (20%, estratificada);
- `val` dos ficheiros → **teste**, usado uma única vez para reportar.

Normalização: em todas as experiências usa-se **√ → L2** (kernel de Hellinger / power normalization),
porque foi a melhor variante nas Partes 1 e 2.

**Referências principais:**
Jégou & Chum (2012), *Negative evidences and co-occurrences in image retrieval: the benefit of PCA and whitening*, ECCV;
Jégou et al. (2010), VLAD; Perronnin et al. (2010), Improved Fisher Vector;
Manning, Raghavan & Schütze (2008), *Introduction to Information Retrieval*, cap. 8 (P@K, R@K, AP);
documentação do scikit-learn (`PCA`, `LinearSVC`, `average_precision_score`, `f1_score`).

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score, accuracy_score, f1_score
from sklearn.model_selection import train_test_split

from src import utils, encoders

SEED = 42
C = 10
CANDIDATE_CATEGORIES = ["aeroplane", "motorbike", "person", "horse", "car"]


def loadEncodingSubset(category, encoding, subset="train", root_dir="data"):
    base_dir = f"{root_dir}/encodings/{encoding}/"
    p_data = np.load(f"{base_dir}/{category}_{subset}.npz")
    n_data = np.load(f"{base_dir}/background_{subset}.npz")

    hist = np.concatenate([p_data["histograms"], n_data["histograms"]], axis=0)
    names = np.concatenate([p_data["file_names"], n_data["file_names"]], axis=0)
    labels = np.concatenate([np.ones(len(p_data["file_names"]), dtype=np.int32),
                             -np.ones(len(n_data["file_names"]), dtype=np.int32)])
    return hist, labels, names


def load_data(category, encoding, val_fraction=0.2, seed=SEED):
    # Igual ao notebook D–I, mas a seed da partição treino/validação passa a ser um parâmetro
    x, y, names = loadEncodingSubset(category, encoding, subset="train")
    x_test, y_test, names_test = loadEncodingSubset(category, encoding, subset="val")
    x, x_test = x.astype(np.float32), x_test.astype(np.float32)

    x_train, x_val, y_train, y_val = train_test_split(
        x, y, test_size=val_fraction, stratify=y, random_state=seed)

    return {"x_train": x_train, "y_train": y_train, "x_val": x_val, "y_val": y_val,
            "x_test": x_test, "y_test": y_test}


def l2norm(x):
    return x / (np.sqrt(np.sum(x**2, axis=1, keepdims=True)) + 1e-10)


def signed_sqrt(x):
    return np.sign(x) * np.sqrt(np.abs(x))


def remove_spatial_information(x, num_tiles=4):
    # Layout de encode_tiling: [tile0 | tile1 | tile2 | tile3] -> média dos 4 tiles
    return x.reshape(x.shape[0], num_tiles, -1).mean(axis=1)


def prepare(x, representation="global", use_sqrt=True):
    # representation: "global" (histograma da imagem inteira) ou "tiling" (4 tiles concatenados)
    if representation == "global":
        x = remove_spatial_information(x)
    if use_sqrt:
        x = signed_sqrt(x)
    return l2norm(x)


def encoding_exists(encoding, category="motorbike"):
    return os.path.exists(f"data/encodings/{encoding}/{category}_train.npz")


categories = [c for c in CANDIDATE_CATEGORIES if encoding_exists("bovw_K512_C0", c)]
print("Categorias disponíveis:", categories)

# Parte 3 — Vocabulário grande + PCA vs. vocabulário pequeno

## Desenho experimental

| Condição | Encoding | PCA | Objetivo |
|---|---|---|---|
| A | BoVW K=256 | não | baseline barata |
| B | BoVW K=512 e K=1024 | não | baselines de maior capacidade |
| C | BoVW K=1024 | sim: 512, 256, 128, 64, 32 | comprimir a representação rica |
| D | vocabulário pequeno com a dimensão de C | não | K=256 já é o "D" de PCA 256; K=128 (D de PCA 128) só se existir |
| — | VLAD e FV (K=32, D=80, tiling) | sim: 256, 128 | ver se a resposta muda para VLAD/FV |

Decisões (e porquê):

1. **BoVW global (K dimensões)**: a nota de equidade do enunciado compara dimensões assumindo que
   o histograma BoVW tem exatamente K dimensões. VLAD e FV ficam com tiling, como na Parte 2.
2. **√ → L2 antes da PCA**: a PCA é linear; aplicá-la ao espaço de Hellinger mantém a mesma
   geometria que o SVM usa. **Nova L2 depois da PCA**, porque a projeção (com centragem) muda a norma
   e o SVM linear funciona melhor com vetores de norma 1 (Jégou & Chum, 2012).
3. **PCA ajustada só com as imagens de treino** (no modelo final: treino+validação), nunca com
   validação nem teste — senão haveria fuga de informação.
4. Restrição do enunciado: dimensão da PCA < K e ≤ nº de imagens de treino − 1 (≥ 900 aqui).
5. **Estabilidade: 3 seeds** da partição treino/validação (42, 0, 1) na validação. O teste é avaliado
   uma vez (seed 42), com retreino em treino+validação.
6. Métricas: AP/mAP (ranking), e **accuracy e macro-F1** com o limiar natural do SVM, f(x) > 0, como o
   enunciado pede para esta parte.

In [ ]:
def run_condition(category, encoding, pca_dim=0, representation="global", seed=SEED, final=False):
    data = load_data(category, encoding, seed=seed)

    if final:   # modelo final: treino + validação -> teste
        x_fit = np.concatenate([data["x_train"], data["x_val"]])
        y_fit = np.concatenate([data["y_train"], data["y_val"]])
        x_eval, y_eval = data["x_test"], data["y_test"]
    else:       # seleção: treino -> validação
        x_fit, y_fit = data["x_train"], data["y_train"]
        x_eval, y_eval = data["x_val"], data["y_val"]

    x_fit = prepare(x_fit, representation)
    x_eval = prepare(x_eval, representation)
    original_dim = x_fit.shape[1]

    # PCA ajustada só com os dados de treino desta corrida
    pca_time = 0.0
    if pca_dim:
        pca = PCA(n_components=pca_dim, random_state=SEED)
        start = time.perf_counter()
        pca.fit(x_fit)
        pca_time = time.perf_counter() - start
        x_fit = l2norm(pca.transform(x_fit))

    start = time.perf_counter()
    W, b = utils.train_svm(x_fit, y_fit, C=C)
    train_time = time.perf_counter() - start

    # Inferência = (projeção PCA, se houver) + produto interno com W
    start = time.perf_counter()
    if pca_dim:
        x_eval = l2norm(pca.transform(x_eval))
    scores = W @ x_eval.T + b
    inference_ms = (time.perf_counter() - start) * 1000 / len(x_eval)

    y_pred = np.where(scores > 0, 1, -1)
    return {"AP": average_precision_score(y_eval, scores),
            "accuracy": accuracy_score(y_eval, y_pred),
            "macro-F1": f1_score(y_eval, y_pred, average="macro"),
            "dim original": original_dim, "dim": x_fit.shape[1],
            "PCA fit (s)": pca_time, "treino SVM (s)": train_time,
            "inferência (ms/img)": inference_ms}

In [ ]:
SEEDS = [42, 0, 1]

# (nome, encoding, representação, dimensão PCA (0 = sem PCA), família para o gráfico)
conditions = [
    ("A: K=256",            "bovw_K256_C0",  "global", 0,   "BoVW sem PCA"),
    ("B: K=512",            "bovw_K512_C0",  "global", 0,   "BoVW sem PCA"),
    ("B: K=1024",           "bovw_K1024_C0", "global", 0,   "BoVW K=1024 + PCA"),
    ("C: K=1024 + PCA 512", "bovw_K1024_C0", "global", 512, "BoVW K=1024 + PCA"),
    ("C: K=1024 + PCA 256", "bovw_K1024_C0", "global", 256, "BoVW K=1024 + PCA"),
    ("C: K=1024 + PCA 128", "bovw_K1024_C0", "global", 128, "BoVW K=1024 + PCA"),
    ("C: K=1024 + PCA 64",  "bovw_K1024_C0", "global", 64,  "BoVW K=1024 + PCA"),
    ("C: K=1024 + PCA 32",  "bovw_K1024_C0", "global", 32,  "BoVW K=1024 + PCA"),
    ("D: K=128",            "bovw_K128_C0",  "global", 0,   "BoVW sem PCA"),
    ("VLAD",                "vlad_K32_C80",  "tiling", 0,   "VLAD + PCA"),
    ("VLAD + PCA 256",      "vlad_K32_C80",  "tiling", 256, "VLAD + PCA"),
    ("VLAD + PCA 128",      "vlad_K32_C80",  "tiling", 128, "VLAD + PCA"),
    ("FV",                  "fv_K32_C80",    "tiling", 0,   "FV + PCA"),
    ("FV + PCA 256",        "fv_K32_C80",    "tiling", 256, "FV + PCA"),
    ("FV + PCA 128",        "fv_K32_C80",    "tiling", 128, "FV + PCA"),
]

# Saltam-se as condições cujos encodings não existem (por exemplo K=128)
available = []
for condition in conditions:
    if encoding_exists(condition[1]):
        available.append(condition)
    else:
        print(f"{condition[0]}: encodings {condition[1]} em falta, a saltar.")

rows = []
for name, encoding, representation, pca_dim, family in available:
    for category in categories:
        for seed in SEEDS:
            r = run_condition(category, encoding, pca_dim, representation, seed=seed)
            rows.append({"condição": name, "família": family, "categoria": category, "seed": seed,
                         "dim": r["dim"], "AP validação": r["AP"]})

val3 = pd.DataFrame(rows)

mean = val3.pivot_table(index="condição", columns="categoria", values="AP validação", aggfunc="mean", sort=False)
std = val3.pivot_table(index="condição", columns="categoria", values="AP validação", aggfunc="std", sort=False)

val_table = mean.round(3).astype(str) + " ± " + std.round(3).astype(str)
val_table.insert(0, "dim", val3.groupby("condição", sort=False)["dim"].first())
val_table["mAP validação"] = mean.mean(axis=1).round(4)
val_table

**Gráfico**: mAP de validação em função da dimensão final. Para o BoVW K=1024, o ponto mais à direita é
sem PCA; os restantes são PCA 512 → 32. Os vocabulários sem PCA (K=256, 512, 128) aparecem como pontos
à parte, para comparar "vocabulário grande + PCA" com "vocabulário pequeno" à mesma dimensão.

In [ ]:
curve = val3.groupby(["condição", "família"], sort=False).agg(
    dim=("dim", "first"), mAP=("AP validação", "mean")).reset_index()

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for family, group in curve.groupby("família", sort=False):
    group = group.sort_values("dim")
    style = "o" if family == "BoVW sem PCA" else "o-"
    ax.plot(group["dim"], group["mAP"], style, label=family, markersize=6)

ax.set_xscale("log", base=2)
ax.set_xlabel("Dimensão final do vetor (escala log2)")
ax.set_ylabel("mAP de validação (média de 3 seeds)")
ax.set_title("Parte 3: vocabulário grande + PCA vs. vocabulário pequeno")
ax.grid(alpha=0.3)
ax.legend()
plt.savefig("parte3_pca.png", dpi=150, bbox_inches="tight")
plt.show()

## Teste (uma vez, seed 42, retreino em treino + validação)

Inclui as medidas de eficiência pedidas: dimensão original e final, % de dimensões removidas,
tempo de ajuste da PCA, tempo de treino do SVM, inferência por imagem (PCA + score) e armazenamento.

In [ ]:
rows = []
for name, encoding, representation, pca_dim, family in available:
    for category in categories:
        r = run_condition(category, encoding, pca_dim, representation, final=True)
        rows.append({"condição": name, "categoria": category, **r})

test3 = pd.DataFrame(rows)

ap_test = test3.pivot_table(index="condição", columns="categoria", values="AP", sort=False).add_prefix("AP ")
part3_test = test3.groupby("condição", sort=False).agg(
    dim_original=("dim original", "first"),
    dim_final=("dim", "first"),
    accuracy=("accuracy", "mean"),
    macro_F1=("macro-F1", "mean"),
    pca_fit_s=("PCA fit (s)", "mean"),
    treino_svm_s=("treino SVM (s)", "mean"),
    inferencia_ms=("inferência (ms/img)", "mean"),
)
part3_test["% removido"] = 100 * (1 - part3_test["dim_final"] / part3_test["dim_original"])
part3_test["vetor (KB/img)"] = part3_test["dim_final"] * 4 / 1024
part3_test = pd.concat([ap_test, part3_test], axis=1)
part3_test.insert(len(ap_test.columns), "mAP teste", ap_test.mean(axis=1))
part3_test.round(4)

## Tempos de codificação e de ajuste do dicionário

**Codificação** (por imagem): igual ao notebook D–I, em 10 imagens de teste.

**Ajuste do dicionário e da PCA local**: o `train.py` não guarda estes tempos, por isso voltamos a
correr as funções dele (`sampleLocalFeatures`, `train_encoder`) **sem gravar nada** — os encoders
em `data/encoders/` não são alterados. Usa os mesmos parâmetros por omissão do `train.py`
(150 imagens por classe, 400 descritores por imagem, seed 42).

⚠️ Esta célula é lenta (o k-means com K=1024 pode demorar alguns minutos). Põe `RUN_CODEBOOK_TIMING = False`
para a saltar.

In [ ]:
from src.features import computeFeatures

encodings_to_time = sorted({encoding for _, encoding, _, _, _ in available})

sample_names = loadEncodingSubset("motorbike", "bovw_K512_C0", subset="val")[2][:10]
sample_paths = [f"data/images/{name}.jpg" for name in sample_names]

start = time.perf_counter()
features = [computeFeatures(path) for path in sample_paths]
sift_ms = (time.perf_counter() - start) * 1000 / len(features)
print(f"SIFT denso: {sift_ms:.0f} ms por imagem")

encoding_ms = {}
for encoding in encodings_to_time:
    encoder = encoders.ImageEncoder(f"data/encoders/{encoding}.npz")
    start = time.perf_counter()
    for keypoints, descriptors, image_size in features:
        encoder.encode_tiling(keypoints, descriptors, image_size)
    encoding_ms[encoding] = (time.perf_counter() - start) * 1000 / len(features)
    print(f"{encoding:15s} codificação: {encoding_ms[encoding]:.0f} ms por imagem")

In [ ]:
RUN_CODEBOOK_TIMING = True

if RUN_CODEBOOK_TIMING:
    from src.train import sampleLocalFeatures, train_encoder

    np.random.seed(SEED)   # como no train.py (baralha os descritores com np.random)
    df = utils.loadSamplesTable(txt_dir="data/splits", img_dir="data/images", subset="train",
                                per_class=150, random_state=SEED)

    start = time.perf_counter()
    descs = sampleLocalFeatures(df, num_samples=400, seed=SEED, cache_dir="cache/")
    sampling_s = time.perf_counter() - start
    print(f"{descs.shape[0]} descritores amostrados em {sampling_s:.1f} s")

    # PCA local (128 -> 80) sozinha, para ter o tempo separado do dicionário
    start = time.perf_counter()
    PCA(n_components=80, random_state=SEED).fit(descs)
    local_pca_s = time.perf_counter() - start

    timing_rows = [{"etapa": "PCA local 128 → 80", "tempo (s)": local_pca_s}]
    for encoder_type, num_words, pca_dim in [("bovw", 256, 0), ("bovw", 512, 0), ("bovw", 1024, 0),
                                            ("vlad", 32, 80), ("fv", 32, 80)]:
        start = time.perf_counter()
        train_encoder(descs, encoder_type=encoder_type, num_words=num_words, pca_dim=pca_dim, seed=SEED)
        label = f"{encoder_type} K={num_words}" + (" (inclui PCA local)" if pca_dim else "")
        timing_rows.append({"etapa": label, "tempo (s)": time.perf_counter() - start})

    codebook_times = pd.DataFrame(timing_rows).round(2)
    display(codebook_times)

# Parte 5 — Retrieval por imagem de consulta (query-by-example)

Aqui **não há classificador**: dada uma imagem de consulta, calcula-se a semelhança com todas as
imagens do corpus e ordena-se. Comparamos as estratégias 1 (BoVW) e 2 (VLAD / Fisher Vector) do enunciado.

## Desenho

- **Categoria-alvo**: motorbike (alterável em `RETRIEVAL_CATEGORY`).
- **Corpus fixo**: todas as imagens do conjunto de teste (`val` dos ficheiros) de todas as categorias
  (aeroplane, motorbike, person) e background, **sem duplicados**. Aeroplane e person funcionam como
  distratores "difíceis" (veículos, pessoas em motos, estradas).
- **Ground truth**: uma imagem do corpus é relevante se estiver na lista de motorbike do teste.
- **Consultas**: 10 imagens de motorbike do **treino** (seed fixa). Treino e teste são partições
  disjuntas, por isso nenhuma consulta está no corpus (verifica-se com um `assert`). Não conseguimos
  garantir a ausência de *near-duplicates*; fica como limitação.
- **Vocabulários/GMM** e **PCA**: aprendidos só com imagens de treino, sem etiquetas.
- **Semelhança**: cosseno (vetores com norma L2 = 1, logo é o produto interno).
- **Métricas**: AP por consulta e mAP; P@5, P@10, P@20; R@20 e R@100 (o número total de
  relevantes no corpus é conhecido); tempo de ranking por consulta e memória do corpus.

In [ ]:
RETRIEVAL_CATEGORY = "motorbike"
N_QUERIES = 10


def load_all(encoding, subset):
    # Todas as imagens de um subset (categorias + background), sem duplicados
    xs, names = [], []
    for category in categories + ["background"]:
        data = np.load(f"data/encodings/{encoding}/{category}_{subset}.npz")
        xs.append(data["histograms"])
        names.extend(data["file_names"])
    x = np.concatenate(xs).astype(np.float32)
    names = np.array(names)

    _, first = np.unique(names, return_index=True)   # 1.ª ocorrência de cada nome
    keep = np.sort(first)                             # mantém a ordem original
    return x[keep], names[keep]


def category_names(category, subset):
    return np.load(f"data/encodings/bovw_K512_C0/{category}_{subset}.npz")["file_names"]


# Consultas: imagens da categoria-alvo no treino
rng = np.random.default_rng(SEED)
query_pool = np.sort(category_names(RETRIEVAL_CATEGORY, "train"))
query_names = rng.choice(query_pool, size=N_QUERIES, replace=False)

_, corpus_names = load_all("bovw_K512_C0", "val")
relevant = np.isin(corpus_names, category_names(RETRIEVAL_CATEGORY, "val"))
assert not np.isin(query_names, corpus_names).any(), "uma consulta está no corpus"

print(f"Corpus: {len(corpus_names)} imagens, {relevant.sum()} relevantes ({relevant.mean():.1%})")

fig, axes = plt.subplots(1, N_QUERIES, figsize=(2 * N_QUERIES, 2.4))
for ax, name in zip(axes, query_names):
    ax.imshow(Image.open(f"data/images/{name}.jpg"))
    ax.set_title(name, fontsize=8)
    ax.axis("off")
plt.suptitle("Imagens de consulta")
plt.savefig("parte5_consultas.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def evaluate_retrieval(encoding, representation="tiling", use_sqrt=True, pca_dim=0):
    corpus_x, names = load_all(encoding, "val")
    train_x, train_names = load_all(encoding, "train")
    assert np.array_equal(names, corpus_names), "ordem do corpus diferente entre encodings"

    row_of = {name: i for i, name in enumerate(train_names)}
    query_x = train_x[[row_of[name] for name in query_names]]

    corpus_x = prepare(corpus_x, representation, use_sqrt)
    query_x = prepare(query_x, representation, use_sqrt)

    pca_time = 0.0
    if pca_dim:
        pca = PCA(n_components=pca_dim, random_state=SEED)
        start = time.perf_counter()
        pca.fit(prepare(train_x, representation, use_sqrt))   # só imagens de treino, sem etiquetas
        pca_time = time.perf_counter() - start
        corpus_x = l2norm(pca.transform(corpus_x))
        query_x = l2norm(pca.transform(query_x))

    # Ranking: produto interno = cosseno, porque todos os vetores têm norma 1
    start = time.perf_counter()
    scores = query_x @ corpus_x.T
    rankings = np.argsort(-scores, axis=1)
    query_ms = (time.perf_counter() - start) * 1000 / len(query_x)

    rows = []
    for i, name in enumerate(query_names):
        hits = relevant[rankings[i]]          # True/False ao longo do ranking
        row = {"consulta": name, "AP": average_precision_score(relevant, scores[i])}
        for k in (5, 10, 20):
            row[f"P@{k}"] = hits[:k].mean()
        for k in (20, 100):
            row[f"R@{k}"] = hits[:k].sum() / relevant.sum()
        rows.append(row)

    costs = {"dim": corpus_x.shape[1], "PCA fit (s)": pca_time,
             "ranking (ms/consulta)": query_ms, "corpus (MB)": corpus_x.nbytes / 1e6}
    return pd.DataFrame(rows), rankings, costs


# (nome, encoding, representação, usar raiz, dimensão PCA)
methods = [
    ("BoVW K=512, L2 (sem raiz)",    "bovw_K512_C0",  "tiling", False, 0),
    ("BoVW K=256",                   "bovw_K256_C0",  "tiling", True,  0),
    ("BoVW K=512",                   "bovw_K512_C0",  "tiling", True,  0),
    ("BoVW K=1024",                  "bovw_K1024_C0", "tiling", True,  0),
    ("BoVW K=512 global",            "bovw_K512_C0",  "global", True,  0),
    ("BoVW K=1024 + PCA 256",        "bovw_K1024_C0", "tiling", True,  256),
    ("BoVW K=1024 + PCA 128",        "bovw_K1024_C0", "tiling", True,  128),
    ("VLAD",                         "vlad_K32_C80",  "tiling", True,  0),
    ("VLAD + PCA 256",               "vlad_K32_C80",  "tiling", True,  256),
    ("VLAD + PCA 128",               "vlad_K32_C80",  "tiling", True,  128),
    ("FV",                           "fv_K32_C80",    "tiling", True,  0),
    ("FV + PCA 256",                 "fv_K32_C80",    "tiling", True,  256),
    ("FV + PCA 128",                 "fv_K32_C80",    "tiling", True,  128),
]

per_query, rankings_by_method, summary_rows = {}, {}, []
for name, encoding, representation, use_sqrt, pca_dim in methods:
    if not encoding_exists(encoding):
        print(f"{name}: encodings em falta, a saltar.")
        continue
    table, rankings, costs = evaluate_retrieval(encoding, representation, use_sqrt, pca_dim)
    per_query[name] = table
    rankings_by_method[name] = rankings
    summary_rows.append({"método": name, "mAP": table["AP"].mean(), "AP desvio": table["AP"].std(),
                         **table[["P@5", "P@10", "P@20", "R@20", "R@100"]].mean().to_dict(), **costs})

part5 = pd.DataFrame(summary_rows).set_index("método")
part5.round(3)

**Sensibilidade à consulta**: AP de cada consulta em cada método. Consultas parecidas com a maioria
das motos do corpus devem ter AP alta em todos os métodos; consultas "atípicas" (escala, ponto de
vista, fundo) devem ter AP baixa.

In [ ]:
ap_by_query = pd.DataFrame({name: table.set_index("consulta")["AP"] for name, table in per_query.items()})
ap_by_query.round(3)

**Exemplos de rankings**: para o melhor método (maior mAP), mostra-se a consulta com maior AP
(sucesso) e a com menor AP (falha), com as 10 primeiras imagens do corpus.
Moldura verde = relevante; vermelha = irrelevante.

In [ ]:
def show_ranking(method, query_name, k=10, file_name=None):
    i = list(query_names).index(query_name)
    top = rankings_by_method[method][i][:k]
    ap = per_query[method].set_index("consulta").loc[query_name, "AP"]

    fig, axes = plt.subplots(1, k + 1, figsize=(1.9 * (k + 1), 2.4))
    axes[0].imshow(Image.open(f"data/images/{query_name}.jpg"))
    axes[0].set_title("consulta", fontsize=9)
    for ax, idx in zip(axes[1:], top):
        ax.imshow(Image.open(f"data/images/{corpus_names[idx]}.jpg"))
        for spine in ax.spines.values():
            spine.set_color("green" if relevant[idx] else "red")
            spine.set_linewidth(4)
    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(f"{method} | consulta {query_name} | AP = {ap:.3f}", fontsize=10)
    if file_name:
        plt.savefig(file_name, dpi=150, bbox_inches="tight")
    plt.show()


best_method = part5["mAP"].idxmax()
aps = per_query[best_method].set_index("consulta")["AP"]
print("Melhor método:", best_method)

show_ranking(best_method, aps.idxmax(), file_name="parte5_sucesso.png")
show_ranking(best_method, aps.idxmin(), file_name="parte5_falha.png")

# A mesma consulta difícil com o BoVW, para comparar estratégias
show_ranking("BoVW K=512", aps.idxmin(), file_name="parte5_falha_bovw.png")

**Não realizado**: as estratégias 3 (CNN/DINOv2) e 4 (CLIP texto → imagem) da Parte 5 são opcionais
("pelo menos duas") e ficaram de fora, tal como a Parte 4.